# 실습 7: 종합 미니 프로젝트 — 문서 자동 분류 & 정보 추출 시스템
**소요시간: 30분** | 난이도: ⭐⭐⭐⭐⭐

## 프로젝트 개요
지금까지 배운 모든 Textract API를 조합하여 **문서를 자동으로 분류하고 정보를 추출하는 시스템**을 만듭니다.

```
입력 문서
    ↓
[1단계] 문서 유형 자동 감지
    ├── 영수증/인보이스 → analyze_expense
    ├── 신분증         → analyze_id
    └── 일반 문서      → analyze_document (FORMS + QUERIES)
    ↓
[2단계] 유형별 정보 추출
    ↓
[3단계] 구조화된 JSON 출력
```

## 🎯 목표
- 새로운 문서를 넣으면 자동으로 분류 후 적절한 API를 선택하여 정보를 추출
- 결과를 통일된 JSON 형식으로 반환


In [1]:
# ✅ [제공 코드] 기본 환경 및 유틸리티
import boto3, os, json, re
from datetime import datetime

textract = boto3.client('textract', region_name='ap-northeast-2')

def load_document_bytes(path):
    with open(path, 'rb') as f:
        return f.read()

def get_text_from_block(block, block_map):
    text = ''
    for rel in block.get('Relationships', []):
        if rel['Type'] == 'CHILD':
            for cid in rel['Ids']:
                child = block_map.get(cid, {})
                if child.get('BlockType') == 'WORD':
                    text += child.get('Text', '') + ' '
    return text.strip()

print("✅ 환경 설정 완료")


✅ 환경 설정 완료


## ✏️ STEP 1: 문서 유형 감지 함수

먼저 `detect_document_text`로 텍스트를 추출한 후, 키워드 기반으로 문서 유형을 판단하세요.


In [2]:
# ✏️ STEP 1: 문서 유형을 감지하는 함수를 완성하세요
def detect_document_type(doc_bytes):
    """
    반환값: 'EXPENSE', 'IDENTITY', 'GENERAL' 중 하나
    """
    # 텍스트 추출
    response = textract.detect_document_text(
        Document={'Bytes': doc_bytes}
    )
    
    # 모든 텍스트를 소문자로 합치기
    all_text = ' '.join([
        b.get('Text', '').lower()
        for b in response['Blocks']
        if b['BlockType'] == 'LINE'
    ])
    
    # 키워드 점수 계산
    expense_keywords = ['receipt', 'invoice', 'total', 'subtotal', 'tax',
                        'amount', 'payment', '영수증', '합계', '세금']
    identity_keywords = ['passport', 'license', 'date of birth', 'expiry',
                         'nationality', 'id number', '여권', '면허']
    
    # 각 유형의 키워드 점수를 계산하세요
    expense_score  = sum(1 for kw in expense_keywords if kw in all_text)   # ← expense_keywords
    identity_score = sum(1 for kw in identity_keywords if kw in all_text)   # ← identity_keywords
    
    print(f"  키워드 점수 — 영수증: {expense_score}, 신분증: {identity_score}")
    
    # 점수 기반 유형 결정
    if identity_score >= 2:
        return 'IDENTITY'    # ← 'IDENTITY'
    elif expense_score >= 2:
        return 'EXPENSE'    # ← 'EXPENSE'
    else:
        return 'GENERAL'    # ← 'GENERAL'

print("✅ detect_document_type() 함수 정의 완료")


✅ detect_document_type() 함수 정의 완료


## ✏️ STEP 2: 유형별 처리 함수

각 문서 유형에 맞는 처리 함수를 작성하세요. 앞 실습의 코드를 참고하세요!


In [3]:
# ✏️ STEP 2A: 영수증 처리 함수
def process_expense(doc_bytes):
    response = textract.analyze_expense(Document={'Bytes': doc_bytes})   # ← analyze_expense
    
    result = {'type': 'EXPENSE', 'data': {}}
    
    if response.get('ExpenseDocuments'):
        doc = response['ExpenseDocuments'][0]
        
        # SummaryFields 파싱
        for field in doc.get('SummaryFields', []):
            key = field.get('Type', {}).get('Text',  '')           # ← 'Text'
            val = field.get('ValueDetection', {}).get('Text',  '') # ← 'Text'
            if key:
                result['data'][key] = val
        
        # LineItems 파싱
        items = []
        for group in doc.get('LineItemGroups', []):
            for item in group.get('LineItems', []):
                item_data = {}
                for field in item.get('LineItemExpenseFields', []):
                    k = field.get('Type', {}).get('Text', '')
                    v = field.get('ValueDetection', {}).get('Text', '')
                    if k: item_data[k] = v
                if item_data: items.append(item_data)
        result['line_items'] = items
    
    return result

print("✅ process_expense() 완료")


✅ process_expense() 완료


In [4]:
# ✏️ STEP 2B: 신분증 처리 함수
def process_identity(doc_bytes):
    response = textract.analyze_id(                     # ← analyze_id
        DocumentPages=[{'Bytes': doc_bytes}]
    )
    
    result = {'type': 'IDENTITY', 'data': {}}
    
    if response.get('IdentityDocuments'):
        id_doc = response['IdentityDocuments'][0]
        for field in id_doc.get('IdentityDocumentFields', []):
            key = field.get('Type', {}).get('Text',  '')           # ← 'Text'
            val = field.get('ValueDetection', {}).get('Text', '')
            norm = field.get('ValueDetection', {}).get('NormalizedValue', {}).get('Value', '')
            result['data'][key] = norm if norm else val
    
    return result

print("✅ process_identity() 완료")


✅ process_identity() 완료


In [5]:
# ✏️ STEP 2C: 일반 문서 처리 함수 (FORMS + QUERIES)
def process_general(doc_bytes, custom_queries=None):
    # 기본 쿼리
    if custom_queries is None:
        custom_queries = [
            {'Text': 'What is the date?', 'Alias': 'DATE'},
            {'Text': 'What is the name?', 'Alias': 'NAME'},
            {'Text': 'What is the reference number?', 'Alias': 'REF_NUM'},
        ]
    
    response = textract.analyze_document(
        Document={'Bytes': doc_bytes},
        FeatureTypes=['FORMS', 'QUERIES'],
        QueriesConfig={'Queries': custom_queries}
    )
    
    blocks = response['Blocks']
    block_map = {b['Id']: b for b in blocks}
    result = {'type': 'GENERAL', 'forms': {}, 'queries': {}}
    
    # FORMS 파싱
    for block in blocks:
        if block['BlockType'] == 'KEY_VALUE_SET' and 'KEY' in block.get('EntityTypes', []):
            key_text = get_text_from_block(block, block_map)
            for rel in block.get('Relationships', []):
                if rel['Type'] == 'VALUE':                          # ← 'VALUE'
                    for vid in rel['Ids']:
                        vblock = block_map.get(vid)
                        if vblock:
                            result['forms'][key_text] = get_text_from_block(vblock, block_map)
    
    # QUERIES 파싱
    for block in blocks:
        if block['BlockType'] == 'QUERY':                           # ← 'QUERY'
            alias = block.get('Query', {}).get('Alias', '')
            for rel in block.get('Relationships', []):
                if rel['Type'] == 'ANSWER':
                    for rid in rel['Ids']:
                        rb = block_map.get(rid)
                        if rb:
                            result['queries'][alias] = rb.get('Text', '')
    
    return result

print("✅ process_general() 완료")


✅ process_general() 완료


## ✏️ STEP 3: 통합 파이프라인 함수

모든 단계를 묶는 최종 함수를 완성하세요.


In [6]:
# ✏️ STEP 3: 통합 파이프라인 함수를 완성하세요
def document_ai_pipeline(file_path, custom_queries=None):
    """
    문서 파일 경로를 받아 자동 분류 후 정보 추출
    반환: {'doc_type': ..., 'file': ..., 'extracted': {...}, 'processed_at': ...}
    """
    print(f"\n{'='*60}")
    print(f"문서 처리: {os.path.basename(file_path)}")
    print(f"{'='*60}")
    
    doc_bytes = load_document_bytes(file_path)                         # ← load_document_bytes
    
    # STEP 1: 문서 유형 감지
    print("\n[1단계] 문서 유형 감지...")
    doc_type = detect_document_type(doc_bytes)                          # ← detect_document_type
    print(f"  → 감지된 유형: {doc_type}")
    
    # STEP 2: 유형별 처리
    print("\n[2단계] 정보 추출...")
    if doc_type == 'EXPENSE':                               # ← 'EXPENSE'
        extracted = process_expense(doc_bytes)
    elif doc_type == 'IDENTITY':                             # ← 'IDENTITY'
        extracted = process_identity(doc_bytes)
    else:
        extracted = process_general(doc_bytes, custom_queries)
    
    result = {
        'doc_type': doc_type,
        'file': os.path.basename(file_path),
        'extracted': extracted,
        'processed_at': datetime.now().isoformat()
    }
    
    print(f"\n✅ 처리 완료")
    print(json.dumps(extracted, ensure_ascii=False, indent=2)[:500])
    
    return result

print("\n✅ document_ai_pipeline() 완료")
print("\n사용법:")
print("  result = document_ai_pipeline('./images/your_document.jpg')")



✅ document_ai_pipeline() 완료

사용법:
  result = document_ai_pipeline('./images/your_document.jpg')


In [7]:
# ✅ [제공 코드] 테스트 실행
# 준비된 이미지 파일로 테스트해보세요

test_files = [
    './images/lab04_receipt.jpg',   # 영수증
    './images/lab05_id.jpg',        # 신분증
    './images/lab02_form.jpg',      # 일반 폼
]

all_results = []
for file_path in test_files:
    if os.path.exists(file_path):
        result = document_ai_pipeline(file_path)
        all_results.append(result)
    else:
        print(f"\n⚠️  파일 없음: {file_path}")

# 결과 저장
if all_results:
    with open('pipeline_results.json', 'w', encoding='utf-8') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    print(f"\n✅ pipeline_results.json 저장 완료 ({len(all_results)}개 문서)")



문서 처리: lab04_receipt.jpg

[1단계] 문서 유형 감지...


  키워드 점수 — 영수증: 2, 신분증: 0
  → 감지된 유형: EXPENSE

[2단계] 정보 추출...



✅ 처리 완료
{
  "type": "EXPENSE",
  "data": {
    "INVOICE_RECEIPT_ID": "35678-9",
    "TOTAL": "2100.00"
  },
  "line_items": [
    {
      "ITEM": "Furniture (Desks and\nChairs)",
      "OTHER": "Merchant One",
      "PRICE": "1500.00",
      "EXPENSE_ROW": "Furniture (Desks and Office Supplies 5/10/1019 Merchant One 1500.00\nChairs)"
    },
    {
      "ITEM": "Team Lunch",
      "OTHER": "Merchant Two",
      "PRICE": "100.00",
      "EXPENSE_ROW": "Team Lunch Food 5/11/2019 Merchant Two 100.00"
    },

문서 처리: lab05_id.jpg

[1단계] 문서 유형 감지...


  키워드 점수 — 영수증: 0, 신분증: 1
  → 감지된 유형: GENERAL

[2단계] 정보 추출...



✅ 처리 완료
{
  "type": "GENERAL",
  "forms": {
    "Other": "",
    "mm dd yy": "/ /",
    "Patient number (medical record or IIS record number)": "012345abcd67",
    "Last Name": "Mary",
    "MI": "M",
    "Date of Birth": "1/6/58",
    "First Name": "Major",
    "Pfizer": "2/8/2021"
  },
  "queries": {
    "DATE": "1/6/58",
    "NAME": "Major"
  }
}

문서 처리: lab02_form.jpg

[1단계] 문서 유형 감지...


  키워드 점수 — 영수증: 0, 신분증: 0
  → 감지된 유형: GENERAL

[2단계] 정보 추출...



✅ 처리 완료
{
  "type": "GENERAL",
  "forms": {
    "Full Name:": "Jane Doe",
    "Home Address:": "123 Any Street, Any Town, USA",
    "Mailing Address:": "same as home address",
    "Phone Number:": "555-0100"
  },
  "queries": {
    "NAME": "Jane Doe"
  }
}

✅ pipeline_results.json 저장 완료 (3개 문서)


## 🏆 과정 완료 체크리스트

아래 항목들을 모두 이해하고 구현했는지 확인하세요!

| 실습 | API | 핵심 개념 | 완료 |
|---|---|---|---|
| Lab 1 | `detect_document_text` | Block 구조, LINE/WORD 계층, BoundingBox | ☐ |
| Lab 2 | `analyze_document` (FORMS/TABLES) | KEY_VALUE_SET, TABLE/CELL 파싱, DataFrame | ☐ |
| Lab 3 | `analyze_document` (QUERIES) | 자연어 쿼리, ANSWER 관계 탐색 | ☐ |
| Lab 4 | `analyze_expense` | SummaryFields, LineItemGroups, 금액 검증 | ☐ |
| Lab 5 | `analyze_id` | IdentityDocumentFields, NormalizedValue, 마스킹 | ☐ |
| Lab 6 | `start_document_analysis` | S3, JobId, 폴링, NextToken 페이지네이션 | ☐ |
| Lab 7 | 종합 | 자동 분류 파이프라인 설계 | ☐ |

## 🚀 다음 단계
- **Textract + Bedrock**: 추출된 텍스트를 Claude/Titan에 전달하여 요약, 분류, Q&A 구현
- **Textract + Comprehend**: 의료/법률 문서에서 개체명(NER) 추출
- **A2I (Augmented AI)**: 신뢰도 낮은 결과에 사람 검토 워크플로우 추가
- **서버리스 파이프라인**: S3 + Lambda + Textract + DynamoDB 아키텍처
